# SFISTA sparse recovery training loss comparison (PyTorch)

Not MMV: infer one sparse vector `x` directly from `y` and `A`. No measurement-column vectorization.

Goal: compare training losses:

1. supervised MSE: `||x_hat - x_true||_2^2`
2. supervised NNMSE: `||x_hat - x_true||_2^2 / (||x_true||_2^2 + eps)`
3. unsupervised LASSO objective: `(1 / (2 n)) * ||A x_hat - y||_2^2 + lam * ||x_hat||_1`, where `n = len(y)`
4. unsupervised cross-validation with fixed lambda: solve on a random per-sample measurement split, train with `(1 / (2 n_val)) * ||A_val x_hat - y_val||_2^2`, using the model's default fixed `lam`
5. unsupervised cross-validation with trainable lambda: same CV loss, but train `lam` together with the unfolded step sizes

For LASSO/CV training, normalize each sample so that `||A.T @ y||_inf = 1`. With the normalized objective, the zero-solution threshold is then `lam_max = 1 / n` for full-data solves, so fixed LASSO values are specified as fractions of `lam_max`: `0.2 / M` and `0.8 / M`. The same scale is applied to `x_true` for test metrics.

For cross-validation training, each batch sample gets an independent random train/validation row split of `A` and `y`. Test metrics use the full measurement set with the same normalization. SFISTA keeps each trainable step size unnormalized and instead scales the shrinkage threshold by the current number of measurements (`theta_k = lam * gamma_k * n`, or `* n_train` for CV splits), with `gamma_init = 1 / ||A||_2^2`. The notebook includes both `cv_fixed_lam` (train unfolded step sizes only) and `cv_lam` (train unfolded step sizes and `lam`). Trainable `lam` is stored directly (no exponential/softplus reparameterization) and clamped to stay non-negative after optimizer steps.


In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

import sys
from pathlib import Path
from time import time

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "src"))
from fa2026.optim import SFISTA

In [2]:
np.random.seed(2)
torch.manual_seed(2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_iter = 6
lr = 1e-3
batch_size = 128
test_size = 1024
N = 30  # dim(x)
M = 15  # dim(y)

# objective
cv_fixed_lam_loss_name = "cv_fixed_lam"
cv_lam_loss_name = "cv_lam"
cv_loss_names = {cv_fixed_lam_loss_name, cv_lam_loss_name}
cv_train_ratio = 0.8
cv_train_size = min(max(int(round(cv_train_ratio * M)), 1), M - 1)

# After normalize_lasso_batch, ||A.T @ y||_inf = 1. For the normalized
# objective (1 / (2 M)) ||A x - y||^2 + lam ||x||_1, lam_max = 1 / M.
lasso_lams = {
    "lasso_0.2_lmax": 0.2 / M,
    "lasso_0.8_lmax": 0.8 / M,
}

# CV inner solves use n_train measurements, so initialize on the same scale.
lam_init = 0.2 / cv_train_size


## Data

`x`: `(batch, N)`, `y`: `(batch, M)`, `A`: `(M, N)`.

In [3]:

p = 0.1
noise_std = 1.0

A = torch.randn(M, N, device=device)


def make_batch(size=batch_size):
    support = (torch.rand(size, N, device=device) < p).float()
    x = torch.randn(size, N, device=device) * support
    y = x @ A.T + noise_std * torch.randn(size, M, device=device)
    return y, x


y, x = make_batch()
print("x", tuple(x.shape), "y", tuple(y.shape), "A", tuple(A.shape))


x (128, 30) y (128, 15) A (15, 30)


## Model

In [4]:

L = torch.linalg.norm(A, ord=2) ** 2
gamma_init = float((1.0 / L).detach().cpu())


def make_model(trainable_lam=True, lam_value=lam_init):
    model = SFISTA(
        n_iterations=N_iter,
        gamma_init=gamma_init,
        norm_A=False,
        positive=False,
    ).to(device)
    lam_tensor = torch.tensor(float(lam_value), device=device)
    if trainable_lam:
        # Store lambda itself, not exp(lambda) / softplus(lambda).
        # Used only by supervised baselines; fixed-lambda methods register a buffer.
        # Positivity is enforced by clamp_lam_ after optimizer steps.
        model.lam = torch.nn.Parameter(lam_tensor.clone())
    else:
        model.register_buffer("lam", lam_tensor)
    return model


print("gamma_init", gamma_init)
print("lasso_lams", lasso_lams)
print("cv lam_init", lam_init)
print("trainable params (supervised)", sum(p.numel() for p in make_model().parameters()))
print("trainable params (fixed-lam LASSO)", sum(p.numel() for p in make_model(trainable_lam=False).parameters()))


gamma_init 0.015081537887454033
lasso_lams {'lasso_0.2_lmax': 0.013333333333333334, 'lasso_0.8_lmax': 0.05333333333333334}
cv lam_init 0.016666666666666666
trainable params (supervised) 7
trainable params (fixed-lam LASSO) 6


## Losses

In [5]:
def current_lam(model):
    # Return a Tensor expression, not the Parameter object itself.
    # Passing the raw Parameter into SFISTA would make its temporary
    # _current_lam_value attribute get registered as a second parameter.
    return model.lam + 0.0


def clamp_lam_(model):
    if getattr(model.lam, "requires_grad", False):
        with torch.no_grad():
            model.lam.clamp_(min=0.0)


def forward_sfista(model, y, train_iter, A_batch=None, lam=None):
    if A_batch is None:
        A_batch = A
    if lam is None:
        lam = current_lam(model)
    return model(y, t=train_iter, A=A_batch, lam=lam)


def mse_loss(x_hat, x_true):
    return ((x_hat - x_true) ** 2).sum(dim=1).mean()


def nnmse_loss(x_hat, x_true, eps=1e-12):
    err = ((x_hat - x_true) ** 2).sum()
    denom = (x_true ** 2).sum().clamp_min(eps)
    return err / denom


def normalize_lasso_batch(y, x, eps=1e-12):
    # After this normalization, full-data lam_max for
    # (1 / (2 M)) ||A x - y||^2 + lam ||x||_1 is 1 / M.
    scale = (y @ A).abs().amax(dim=1, keepdim=True).clamp_min(eps)
    return y / scale, x / scale, scale


def lasso_objective(x_hat, y, lam):
    residual = x_hat @ A.T - y
    n_meas = y.shape[1]
    data_fit = 0.5 * (residual ** 2).sum(dim=1) / n_meas
    sparsity = torch.abs(x_hat).sum(dim=1)
    return (data_fit + lam * sparsity).mean()


def make_cv_split(y, train_ratio=cv_train_ratio):
    n_train = int(round(train_ratio * M))
    n_train = min(max(n_train, 1), M - 1)
    indices = torch.stack([torch.randperm(M, device=y.device) for _ in range(y.shape[0])])
    train_idx = indices[:, :n_train]
    val_idx = indices[:, n_train:]
    return (
        y.gather(1, train_idx),
        A[train_idx],
        y.gather(1, val_idx),
        A[val_idx],
    )


def cv_objective(x_hat, y_val, A_val):
    residual = torch.bmm(A_val, x_hat.unsqueeze(-1)).squeeze(-1) - y_val
    n_val = y_val.shape[1]
    return (0.5 * (residual ** 2).sum(dim=1) / n_val).mean()

supervised_loss_fns = {
    "mse": mse_loss,
    "nnmse": nnmse_loss,
}

def to_db(value):
    return 10 * np.log10(max(float(value), 1e-12))


## Train / evaluate

In [ ]:
def evaluate(model, y, x, train_iter=N_iter, lam=None):
    model.eval()
    with torch.no_grad():
        if lam is None:
            lam = current_lam(model)
        x_hat = forward_sfista(model, y, train_iter, lam=lam)
        metrics = {
            "mse": float(mse_loss(x_hat, x).cpu()),
            "nnmse": float(nnmse_loss(x_hat, x).cpu()),
        }
        metrics["lasso"] = float(lasso_objective(x_hat, y, lam).cpu())
        return metrics


def train(loss_name, max_steps=500, cv_ratio=cv_train_ratio):
    is_lasso = loss_name in lasso_lams
    is_cv = loss_name in cv_loss_names
    is_cv_fixed_lam = loss_name == cv_fixed_lam_loss_name
    fixed_lam = lasso_lams.get(loss_name, lam_init)

    # LASSO and cv_fixed_lam use a fixed model lambda. cv_lam trains lambda.
    model = make_model(trainable_lam=not (is_lasso or is_cv_fixed_lam), lam_value=fixed_lam)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    y_test, x_test = make_batch(test_size)
    if is_lasso:
        y_test, x_test, _ = normalize_lasso_batch(y_test, x_test)

    history = {
        "train_loss": [],
        "test_mse": [],
        "test_nnmse": [],
        "test_lasso": [],
        "lam": [],
    }

    t0 = time()
    for train_iter in range(1, N_iter + 1):
        for step in range(max_steps):
            model.train()
            y_train, x_train = make_batch()
            if is_lasso:
                y_train, x_train, _ = normalize_lasso_batch(y_train, x_train)

            if is_cv:
                y_inner, A_inner, y_val, A_val = make_cv_split(y_train, cv_ratio)
                x_hat = forward_sfista(model, y_inner, train_iter, A_batch=A_inner)
                loss = cv_objective(x_hat, y_val, A_val)
            else:
                x_hat = forward_sfista(model, y_train, train_iter)
                if is_lasso:
                    loss = lasso_objective(x_hat, y_train, current_lam(model))
                else:
                    loss = supervised_loss_fns[loss_name](x_hat, x_train)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            clamp_lam_(model)

            lam_now = current_lam(model).detach()
            metrics = evaluate(model, y_test, x_test, train_iter, lam=lam_now)
            history["train_loss"].append(float(loss.detach().cpu()))
            history["test_mse"].append(metrics["mse"])
            history["test_nnmse"].append(metrics["nnmse"])
            history["test_lasso"].append(metrics["lasso"])
            history["lam"].append(float(lam_now.cpu()))

        msg = (
            f"{loss_name} | iter {train_iter:02d} | "
            f"train {to_db(history['train_loss'][-1]):.2f} dB | "
            f"test NNMSE {to_db(history['test_nnmse'][-1]):.2f} dB"
        )
        if is_lasso or is_cv:
            msg += f" | test LASSO {to_db(history['test_lasso'][-1]):.2f} dB"
        if is_lasso or is_cv:
            msg += f" | lam {history['lam'][-1]:.4g}"
        print(msg)

    final_lam = current_lam(model).detach()
    return {
        "model": model,
        "history": history,
        "seconds": time() - t0,
        "final_lam": float(final_lam.cpu()),
        "final": evaluate(model, y_test, x_test, lam=final_lam),
    }


In [7]:
results = {name: train(name) for name in [cv_fixed_lam_loss_name, cv_lam_loss_name, *lasso_lams.keys(), "mse", "nnmse"]}

for name, result in results.items():
    final = result["final"]
    msg = (
        f"{name}: "
        f"MSE={final['mse']:.4e} ({to_db(final['mse']):.2f} dB), "
        f"NNMSE={final['nnmse']:.4e} ({to_db(final['nnmse']):.2f} dB)"
    )
    if name in lasso_lams or name in cv_loss_names:
        msg += f", LASSO={final['lasso']:.4e} ({to_db(final['lasso']):.2f} dB)"
        msg += f", lam={result['final_lam']:.4g}"
    msg += f", time={result['seconds']:.2f}s"
    print(msg)


cv_fixed_lam | iter 01 | train 2.02 dB | test NNMSE -1.11 dB | test LASSO -24.65 dB | lam 0.01667
cv_fixed_lam | iter 02 | train 2.58 dB | test NNMSE -1.44 dB | test LASSO -24.98 dB | lam 0.01667
cv_fixed_lam | iter 03 | train 2.04 dB | test NNMSE -1.53 dB | test LASSO -25.06 dB | lam 0.01667
cv_fixed_lam | iter 04 | train 2.78 dB | test NNMSE -1.56 dB | test LASSO -25.08 dB | lam 0.01667
cv_fixed_lam | iter 05 | train 2.25 dB | test NNMSE -1.61 dB | test LASSO -25.11 dB | lam 0.01667
cv_fixed_lam | iter 06 | train 1.62 dB | test NNMSE -1.67 dB | test LASSO -25.16 dB | lam 0.01667
cv_lam | iter 01 | train 1.65 dB | test NNMSE 0.00 dB | test LASSO -23.12 dB | lam 0.4418
cv_lam | iter 02 | train 0.88 dB | test NNMSE 0.00 dB | test LASSO -23.12 dB | lam 0.6305
cv_lam | iter 03 | train 1.80 dB | test NNMSE 0.00 dB | test LASSO -23.12 dB | lam 0.588
cv_lam | iter 04 | train 1.72 dB | test NNMSE 0.00 dB | test LASSO -23.12 dB | lam 0.524
cv_lam | iter 05 | train 1.96 dB | test NNMSE 0.00 dB 

KeyboardInterrupt: 

## Curves

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
for name, result in results.items():
    plt.plot([to_db(v) for v in result["history"]["train_loss"]], label=name)
plt.xlabel("training step")
plt.ylabel("training objective [dB]")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
for name, result in results.items():
    plt.plot([to_db(v) for v in result["history"]["test_nnmse"]], label=name)
plt.xlabel("training step")
plt.ylabel("test NNMSE [dB]")
plt.grid(True)
plt.legend()

plt.tight_layout()
